# Event Signal & Generic Data Workflow Notebook

本 Notebook 提供一个通用数据分析骨架，同时可用于后续扩展稀疏事件信号分析（Qlib 环境下）。

## 1. Set Up Environment
说明：导入常用库，设置显示参数；若需要 Qlib，可在此处初始化（后续根据需要取消注释）。

In [ ]:
# 1. Set Up Environment
import os
import math
import json
import time
import gc
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

# 可选：初始化 Qlib (若本地已安装并准备使用)
# import qlib
# from qlib.constant import REG_CN
# if not qlib.is_initialized():
#     qlib.init(mount_path=os.path.expanduser("~/.qlib/qlib_data/cn_data"), region=REG_CN)

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

print("Environment ready.")

## 2. Define Helper Functions
封装通用工具：计时、内存、绘图辅助。

In [ ]:
# 2. Helper Functions
from contextlib import contextmanager

def sizeof_fmt(num, suffix='B'):
    for unit in ['','K','M','G','T','P','E','Z']:
        if abs(num) < 1024.0:
            return f"{num:3.1f}{unit}{suffix}"
        num /= 1024.0
    return f"{num:.1f}Y{suffix}"

@contextmanager
def timer(name: str):
    t0 = time.time()
    yield
    dt = time.time() - t0
    print(f"[TIMER] {name}: {dt:.3f}s")

def mem_usage(obj):
    if hasattr(obj, 'memory_usage'):
        return sizeof_fmt(obj.memory_usage(deep=True).sum())
    return sizeof_fmt(sys.getsizeof(obj))

def plot_hist(df, col, bins=50):
    plt.figure(figsize=(6,3))
    sns.histplot(df[col].dropna(), bins=bins, kde=True)
    plt.title(f"Histogram: {col}")
    plt.show()

def plot_rolling(series, window=20, title=None):
    plt.figure(figsize=(8,3))
    series = series.sort_index()
    roll = series.rolling(window).mean()
    plt.plot(series.index, series.values, label='raw', alpha=0.5)
    plt.plot(roll.index, roll.values, label=f'rolling_mean_{window}')
    plt.legend()
    plt.title(title or f"Rolling Mean ({window})")
    plt.show()

## 3. Load or Simulate Data
若无真实数据，这里构造一个模拟数据集：包含日期、instrument、base_score、event_flag、price。

In [ ]:
# 3. Load or Simulate Data
# 为演示：模拟 250 个交易日、300 支股票、稀疏事件信号（约每日 10 支触发）。
np.random.seed(42)
trading_days = pd.bdate_range('2023-01-01', periods=250)
instruments = [f'STK{i:04d}' for i in range(300)]

# 模拟 base_score (动量或其它打分)
base_scores = []
for d in trading_days:
    scores = np.random.randn(len(instruments))
    base_scores.append(pd.DataFrame({
        'date': d.date(),
        'instrument': instruments,
        'base_score': scores
    }))
base_df = pd.concat(base_scores, ignore_index=True)

# 模拟价格路径 (随机游走)
prices = []
for ins in instruments:
    start = 100 + np.random.randn() * 5
    steps = np.random.randn(len(trading_days)) * 1.0
    path = start + np.cumsum(steps)
    prices.append(pd.DataFrame({'date': trading_days.date, 'instrument': ins, 'close': path}))
price_df = pd.concat(prices, ignore_index=True)

# 稀疏事件：每日选 10 支股票，倾向 base_score 高的股票 (模拟正相关)
events_list = []
for d in trading_days:
    day_scores = base_df[base_df.date == d.date()].sort_values('base_score', ascending=False)
    k = 10
    selected = day_scores.head(k)['instrument'].tolist()
    events_list.append(pd.DataFrame({'date': d.date(), 'instrument': selected, 'event_flag': 1}))
event_df = pd.concat(events_list, ignore_index=True)

print("Simulated base_df rows:", len(base_df))
print("Simulated event_df rows:", len(event_df))
print("Price df rows:", len(price_df))

## 4. Basic Data Inspection
基础查看：结构、缺失值、示例行。

In [ ]:
# 4. Basic Inspection
print(base_df.head())
print(base_df.tail())
print(base_df.dtypes)
print("Missing base_score?", base_df['base_score'].isna().sum())

print(event_df.head())
print("Event days:", event_df['date'].nunique())

# Merge price & base for sanity
merged = base_df.merge(price_df, on=['date','instrument'], how='left')
print("Merged missing prices:", merged['close'].isna().sum())

## 5. Data Cleaning Steps
示例：去重复、确保类型、潜在异常过滤。

In [ ]:
# 5. Data Cleaning
# 去重复
before = len(base_df)
base_df = base_df.drop_duplicates(['date','instrument'])
print("Removed duplicates:", before - len(base_df))

# 类型保证
event_df['event_flag'] = event_df['event_flag'].astype(int)

# 简单异常值裁剪 (base_score 限制在 +/-4 标准差)
std_lim = base_df['base_score'].std() * 4ase_df['base_score'] = base_df['base_score'].clip(-std_lim, std_lim)

print("Cleaning done.")

## 6. Feature Engineering
构造 forward_return、标准化得分、事件延长持仓示例标记。

In [ ]:
# 6. Feature Engineering
# 计算 forward return: 下日收盘 / 当日收盘 - 1
price_pivot = price_df.pivot(index='date', columns='instrument', values='close').sort_index()
forward_ret = price_pivot.shift(-1) / price_pivot - 1

# 标准化 base_score (z-score)
base_df['base_score_z'] = base_df.groupby('date')['base_score'].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))

# 延长持仓事件标记：事件触发后 3 日内 active=1 (示例)
hold_days = 3
event_active_rows = []
for ins in instruments:
    dates_ins = event_df[event_df.instrument == ins]['date'].tolist()
    active_dates = set()
    for d in dates_ins:
        idx = trading_days.get_loc(pd.Timestamp(d)) if pd.Timestamp(d) in trading_days else None
        if idx is None:
            continue
        for offset in range(hold_days):
            if idx + offset < len(trading_days):
                active_dates.add(trading_days[idx + offset].date())
    for ad in active_dates:
        event_active_rows.append({'date': ad, 'instrument': ins, 'event_active': 1})
active_df = pd.DataFrame(event_active_rows)
feat_df = base_df.merge(active_df, on=['date','instrument'], how='left')
feat_df['event_active'] = feat_df['event_active'].fillna(0)
print(feat_df.head())

## 7. Visualization Examples
绘制分布、相关性与滚动统计。

In [ ]:
# 7. Visualization
# base_score 分布
day_sample = base_df[base_df.date == base_df.date.iloc[0]]
plot_hist(day_sample, 'base_score')

# 事件每日数量曲线
evt_count = event_df.groupby('date')['instrument'].count()
plt.figure(figsize=(8,3))
plt.plot(evt_count.index, evt_count.values, label='event_count')
plt.title('Daily Event Count')
plt.show()

# 事件股票 1 日后平均收益 (使用 forward_ret)
ret_event_day1 = []
for d, grp in event_df.groupby('date'):
    if d not in forward_ret.index:
        continue
    r = forward_ret.loc[d, grp['instrument']].mean()
    ret_event_day1.append({'date': d, 'ret1': r})
ret_event_day1_df = pd.DataFrame(ret_event_day1).sort_values('date')
plot_rolling(ret_event_day1_df.set_index('date')['ret1'], window=20, title='Event Day1 Avg Return Rolling Mean')

## 8. Simple Statistical Summary
事件后窗口收益、命中率、与随机对照差异（示例简化）。

In [ ]:
# 8. Statistical Summary
windows = [1,3,5]
summary_rows = []
np.random.seed(123)
all_ins = instruments
for w in windows:
    evt_samples = []
    ctrl_samples = []
    for d, grp in event_df.groupby('date'):
        if d not in forward_ret.index:
            continue
        idx = list(forward_ret.index).index(d)
        seq_dates = forward_ret.index[idx: idx + w]
        if len(seq_dates) < w:
            continue
        insts = grp['instrument'].tolist()
        for ins in insts:
            s = forward_ret.loc[seq_dates, ins]
            evt_samples.append((s + 1).prod() - 1)
        # control same size
        k = len(insts)
        candidates = [x for x in all_ins if x not in insts]
        if len(candidates) >= k:
            sample = np.random.choice(candidates, size=k, replace=False)
            for ins in sample:
                s = forward_ret.loc[seq_dates, ins]
                ctrl_samples.append((s + 1).prod() - 1)
    if evt_samples and ctrl_samples:
        e_arr = np.array(evt_samples)
        c_arr = np.array(ctrl_samples)
        diff_mean = e_arr.mean() - c_arr.mean()
        # 简单标准误
        se = math.sqrt(e_arr.var(ddof=1)/len(e_arr) + c_arr.var(ddof=1)/len(c_arr))
        t_val = diff_mean / se if se > 0 else np.nan
    else:
        diff_mean = np.nan
        t_val = np.nan
    summary_rows.append({'window': w, 'event_mean': np.mean(evt_samples) if evt_samples else np.nan,
                         'control_mean': np.mean(ctrl_samples) if ctrl_samples else np.nan,
                         'diff_mean': diff_mean, 't_value': t_val})
summary_df = pd.DataFrame(summary_rows)
print(summary_df)

## 9. Train/Test Split (Generic Example)
演示如何将 engineered 特征用于一个简单监督学习场景（非事件必需，只作骨架）。

In [ ]:
# 9. Train/Test Split
# 构造一个标签：未来1日收益是否为正 (classification)
label_series = []
for d in trading_days[:-1]:
    if d.date() not in forward_ret.index:
        continue
    day_forward = forward_ret.loc[d.date()]
    for ins in instruments:
        label_series.append({'date': d.date(), 'instrument': ins, 'y': 1 if day_forward.get(ins, 0) > 0 else 0})
label_df = pd.DataFrame(label_series)
model_df = feat_df.merge(label_df, on=['date','instrument'], how='inner')

# 简化选择部分特征
use_cols = ['base_score','base_score_z','event_active']
X = model_df[use_cols]
y = model_df['y']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

## 10. Baseline Model
训练一个简单分类树或逻辑回归。

In [ ]:
# 10. Baseline Model
from sklearn.tree import DecisionTreeClassifier
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
with timer('fit_model'):
    clf.fit(X_train, y_train)
print("Model trained.")

## 11. Model Evaluation Metrics
输出准确率、分类报告；绘制特征重要度。

In [ ]:
# 11. Evaluation
from sklearn.metrics import accuracy_score, classification_report
pred = clf.predict(X_test)
acc = accuracy_score(y_test, pred)
print("Accuracy:", acc)
print(classification_report(y_test, pred))

# Feature importance
imp = pd.Series(clf.feature_importances_, index=use_cols).sort_values(ascending=False)
plt.figure(figsize=(4,3))
imp.plot(kind='bar', title='Feature Importance')
plt.tight_layout()
plt.show()

## 12. Save Artifacts
持久化模型与处理后数据。

In [ ]:
# 12. Save Artifacts
import joblib
artifacts_dir = Path('examples/factor_selection_basic/output/artifacts')
artifacts_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, artifacts_dir / 'baseline_model.joblib')
model_df.head(1000).to_csv(artifacts_dir / 'sample_feature_dataset.csv', index=False)
print("Artifacts saved:", artifacts_dir)